# 02 — Entrenamiento, evaluación y modelo final 2026

Compara baselines, Ridge, Random Forest, Extra Trees, boosting y MLP mediante backtesting expansivo. La selección usa MAE promedio de 2024–2025.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

try:
    from google.colab import drive
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    drive.mount('/content/drive')
    REPO_ROOT = Path('/content/suelosabio')
    if not (REPO_ROOT / '.git').exists():
        subprocess.run([
            'git', 'clone', '--depth', '1', '--branch', 'feature/SCRUM-17',
            'https://github.com/cybercolombia/suelosabio.git', str(REPO_ROOT)
        ], check=True)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '-r',
        str(REPO_ROOT / 'notebooks/CropForecasting/requirements.txt')
    ], check=True)
else:
    REPO_ROOT = Path.cwd()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / '.git').exists():
        REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT))

In [ ]:
from notebooks.CropForecasting.config import load_forecasting_config
from notebooks.CropForecasting.modeling import metric_breakdown, run_temporal_backtesting

CONFIG = load_forecasting_config(in_colab=EN_COLAB, mount_drive=False)
DATASET_ROOT = CONFIG.dataset_root
MODEL_ROOT = CONFIG.model_root

dataset = pd.read_parquet(DATASET_ROOT / 'dataset_definitivo.parquet')
scenarios = pd.read_parquet(DATASET_ROOT / 'escenarios_climaticos_asof.parquet')
manifest = json.loads((DATASET_ROOT / 'manifest.json').read_text(encoding='utf-8'))
climate_features = manifest['climate_features']

## Entrenamiento

Active la bandera para repetir todos los folds y entrenar el modelo final. La corrida guardada se muestra aun cuando la bandera permanece desactivada.

In [ ]:
ENTRENAR_MODELOS = False

if ENTRENAR_MODELOS:
    evaluacion = run_temporal_backtesting(dataset, scenarios, climate_features)
    leaderboard = evaluacion.leaderboard
    metricas_fold = evaluacion.fold_metrics
    predicciones_backtest = evaluacion.backtest_predictions
    pronostico = evaluacion.forecast_2026
else:
    leaderboard = pd.read_csv(MODEL_ROOT / 'leaderboard.csv')
    metricas_fold = pd.read_csv(MODEL_ROOT / 'metricas_por_fold.csv')
    predicciones_backtest = pd.read_parquet(MODEL_ROOT / 'predicciones_backtest.parquet')
    pronostico = pd.read_parquet(MODEL_ROOT / 'pronostico_2026.parquet')

display(leaderboard)

In [ ]:
ganador = leaderboard.iloc[0]
display(pd.DataFrame([{
    'modelo_final': ganador['model'],
    'representacion': ganador['encoding'],
    'MAE_2024_2025': ganador['mae_media'],
    'RMSE_2024_2025': ganador['rmse_media'],
    'R2_2024_2025': ganador['r2_media'],
}]))

display(metricas_fold[metricas_fold.candidate.eq(ganador.candidate)])
display(metric_breakdown(predicciones_backtest))

## Resultado 2026

EVA 2026 todavía no tiene target observado. Estas filas son pronósticos; las métricas anteriores pertenecen exclusivamente al backtesting 2021–2025.

In [ ]:
resumen = (
    pronostico.groupby(['departamento', 'tipo_periodo'])
    .agg(
        municipios=('codigo_municipio', 'size'),
        media_t_ha=('prediccion_rendimiento_t_ha', 'mean'),
        mediana_t_ha=('prediccion_rendimiento_t_ha', 'median'),
        minimo_t_ha=('prediccion_rendimiento_t_ha', 'min'),
        maximo_t_ha=('prediccion_rendimiento_t_ha', 'max'),
    )
    .reset_index()
)
display(resumen)
display(pronostico)